Visualization

In [ ]:
#Connect Database
import duckdb
import pandas as pd

# Connect to an existing DuckDB database
conn = duckdb.connect("data/jobsdb.db")
df = conn.execute("""
    SELECT *
    FROM SGJobData_CleanedAndExploded
""").fetchdf()
conn.close()


,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,minimumYearsExperience,...,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary,parsed_categories
0,Permanent,2023-05-08,False,MCF-2023-0252866,2023-04-08,2023-03-30,2,5,151,0,...,Executive,WORKSTONE PTE. LTD.,2800.0,2000.0,Monthly,0.0,Closed,Food Technologist - Clementi | Entry Level | U...,2400.0,Environment / Health
1,Permanent,2023-05-08,False,MCF-2023-0252866,2023-04-08,2023-03-30,2,5,151,0,...,Executive,WORKSTONE PTE. LTD.,2800.0,2000.0,Monthly,0.0,Closed,Food Technologist - Clementi | Entry Level | U...,2400.0,Manufacturing
2,Permanent,2023-05-08,False,MCF-2023-0252866,2023-04-08,2023-03-30,2,5,151,0,...,Executive,WORKSTONE PTE. LTD.,2800.0,2000.0,Monthly,0.0,Closed,Food Technologist - Clementi | Entry Level | U...,2400.0,Sciences / Laboratory / R&D
3,Permanent,2023-05-08,False,MCF-2023-0273977,2023-04-08,2023-04-08,0,0,55,2,...,Executive,TRUST RECRUIT PTE. LTD.,5500.0,4000.0,Monthly,0.0,Closed,"Software Engineer (Fab Support) (Java, CIM, Up...",4750.0,Information Technology
4,Full Time,2023-04-22,False,MCF-2023-0273994,2023-04-08,2023-04-08,0,7,99,3,...,Senior Executive,PU TIEN SERVICES PTE. LTD.,4600.0,3800.0,Monthly,0.0,Closed,Senior Technician,4200.0,Repair and Maintenance
5,Permanent,2023-05-08,False,MCF-2023-0273991,2023-04-08,2023-04-08,0,6,113,8,...,Senior Executive,TRUST RECRUIT PTE. LTD.,10000.0,5000.0,Monthly,0.0,Closed,"Senior .NET Developer (.NET Core, MVC, MVVC, S...",7500.0,Information Technology
6,Full Time,2023-05-08,False,MCF-2023-0273976,2023-04-08,2023-04-08,0,3,99,2,...,Non-executive,EATZ CATERING SERVICES PTE. LTD.,3400.0,2400.0,Monthly,0.0,Closed,Sales / Admin Cordinator,2900.0,Admin / Secretarial
7,Full Time,2023-04-22,False,MCF-2023-0273974,2023-04-08,2023-04-08,0,4,110,1,...,Junior Executive,BYTECENTURE CONSULTING PTE. LTD.,6000.0,4000.0,Monthly,0.0,Closed,Software Support Engineer,5000.0,Consulting
8,Full Time,2023-04-22,False,MCF-2023-0273974,2023-04-08,2023-04-08,0,4,110,1,...,Junior Executive,BYTECENTURE CONSULTING PTE. LTD.,6000.0,4000.0,Monthly,0.0,Closed,Software Support Engineer,5000.0,Information Technology
9,Full Time,2023-04-22,False,MCF-2023-0273974,2023-04-08,2023-04-08,0,4,110,1,...,Junior Executive,BYTECENTURE CONSULTING PTE. LTD.,6000.0,4000.0,Monthly,0.0,Closed,Software Support Engineer,5000.0,Professional Services


In [32]:
%%writefile app.py
from pathlib import Path

import streamlit as st
import plotly.express as px
import duckdb
import pandas as pd

st.set_page_config(page_title="Singapore Job Market Dashboard", layout="wide")

# Connect to an existing DuckDB database

conn = duckdb.connect("data/jobsdb.db")
df = conn.execute("""
    SELECT *
    FROM SGJobData_CleanedAndExploded limit 100000
""").fetchdf()
conn.close()

st.title("Singapore Job Market Dashboard")

st.header("Overview")

with st.sidebar:
    st.header("Filters")
    category_options = sorted(df["parsed_categories"].dropna().unique())
    selected_categories = st.multiselect(
        "Industry", category_options, default=category_options
    )
  
    
col1, col2, col3 = st.columns(3)
col1.metric("Total Postings", len(df))
col2.metric("Industry", len(df["parsed_categories"].unique()))
# col3.metric(
#      "Average Salary",
#      f"${df['average_salary'].mean():,.0f}" if len(df) else "N/A",
# )
col3.metric("Current Job Openings", df[df["status_jobStatus"] != "Closed"]["numberOfVacancies"].sum())
with st.expander("View raw data"):
    st.dataframe(df)

tab1,tab2 = st.tabs(["Job Vacancy Heatmap", "Job Level Mix"])

with tab1:
    st.subheader("Job Vacancy Heatmap")

    def experience_group(years):
        if years <= 2:
            return "Entry<br>(0-2) Yrs"
        elif years <= 5:
            return "Mid <br>(3-5) Yrs"
        elif years <= 10:
            return "Senior<br>(6-10) Yrs"
        else:
            return "Expert<br>(10+) Yrs"
        
    df["experience_group"] = df["minimumYearsExperience"].map(experience_group)

    # Create a frequency table
    heatmap_data = (
        df.groupby(["parsed_categories", "experience_group", "numberOfVacancies"])
        .size()
        .reset_index(name="count")
    )

    fig = px.density_heatmap(
        heatmap_data,
        x="experience_group",
        y="parsed_categories",
        z="count",
        color_continuous_scale="Blues",
        category_orders={
        "experience_group": ["Entry<br>(0-2) Yrs", "Mid <br>(3-5) Yrs", "Senior<br>(6-10) Yrs", "Expert<br>(10+) Yrs"],
        "parsed_categories": sorted(heatmap_data["parsed_categories"].unique())},
        text_auto=True
    )
    fig.update_layout(
        height=1000,
        xaxis=dict(tickfont=dict(size=10), side ="top"),
        xaxis_title="<b>Experience Level</b>",
        yaxis=dict(tickfont=dict(size=10)),
        yaxis_title="<b>Industry Category</b>",
        coloraxis_colorbar_title="<b>Number of Vacancies</b>"
    )
    
    st.plotly_chart(fig, use_container_width=True)
with tab2:

    st.subheader("Job Level Mix Within Each Industry")
   
    industry_joblevel = (
    df.groupby(["parsed_categories", "experience_group"])
      .size()
      .reset_index(name="numberOfVacancies")
    )

    # Calculate percentage within each industry
    industry_joblevel["Percentage"] = (
    industry_joblevel["numberOfVacancies"]
    / industry_joblevel.groupby("parsed_categories")["numberOfVacancies"].transform("sum")
    * 100
    )
    
    fig = px.bar(
    industry_joblevel,
    x="Percentage",
    y="parsed_categories",
    color="experience_group",
    title="Job Vacancies by Experience Level Within Each Industry",
    labels={
        "parsed_categories": "Industry",
        "Percentage": "Percentage of Vacancies",
        "experience_group": "Experience Level"
    },
    barmode="stack"
)


# fig.update_layout(
#     barmode="stack",
#     xaxis_title="Percentage of Vacancies",
#     yaxis_title="Industry",
#     legend_title="Experience Level",
#     template="plotly_white"
# )
fig.update_layout()
st.plotly_chart(fig, use_container_width=True)


Overwriting app.py
